# Curve-Supervised Hard3 - Google Colab
Once `pseudo_smoke` ile boru hattini dogrulayin. Yayin deneyi icin gercek hairline/jaw curve manifesti gerekir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'
if CODE_ROOT.exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
print(CODE_ROOT)

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/orthodontic/data/dataset')
PILOT_MANIFEST = Path('/content/drive/MyDrive/orthodontic/annotations/hard3_curves_pilot_v2.json')
SPLIT_REPORT = Path('/content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42/fold_1/split_and_leakage_report.json')
assert DATA_ROOT.exists(), f'Dataset bulunamadi: {DATA_ROOT}'
assert SPLIT_REPORT.exists(), f'Fold-1 split raporu bulunamadi: {SPLIT_REPORT}'
print('Dataset:', DATA_ROOT)
print('Pilot manifest exists:', PILOT_MANIFEST.exists())

## 1. Hizli pseudo-curve smoke testi

In [ ]:
%cd /content/comparative-study
!python -u curve_supervised_hard3_refinement/colab_run_curve_hard3.py --preset pseudo_smoke --seed 42

## 2. Dengeli 24-ornek pilot manifesti
Yalniz Fold-1 train orneklerinden secilir ve mevcut dolu manifestin uzerine yazmaz.

In [ ]:
if not PILOT_MANIFEST.exists():
    PILOT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'python', '-u',
        str(CODE_ROOT / 'curve_supervised_hard3_refinement/prepare_annotations.py'),
        '--data-root', str(DATA_ROOT),
        '--split-report', str(SPLIT_REPORT),
        '--split-name', 'train',
        '--sample-count', '24',
        '--seed', '42',
        '--output', str(PILOT_MANIFEST),
    ], check=True, cwd=str(CODE_ROOT))
else:
    print('Mevcut manifest korundu:', PILOT_MANIFEST)

## 3. Gercek curve anotasyonlu pilot
24 secili ornegin hairline, jaw_left ve jaw_right alanlari doldurulduktan sonra calistirin. Pilot hicbir zaman yayin sonucu sayilmaz.

In [ ]:
!python -u curve_supervised_hard3_refinement/colab_run_curve_hard3.py --preset annotated_pilot --seed 42 --annotation-manifest {PILOT_MANIFEST} --pilot-annotated-samples 24

## 4. Yayin Fold-1 kapisi
Yalniz pilot Hard3 sonucunda belirgin OOF iyilesme gorulurse en az 60 outer-train tam anotasyonuyla calistirin.

In [ ]:
PUBLICATION_MANIFEST = Path('/content/drive/MyDrive/orthodontic/annotations/hard3_curves_publication_v2.json')
assert PUBLICATION_MANIFEST.exists(), f'Yayin manifesti bulunamadi: {PUBLICATION_MANIFEST}'
!python -u curve_supervised_hard3_refinement/colab_run_curve_hard3.py --preset annotated_fold1 --seed 42 --annotation-manifest {PUBLICATION_MANIFEST} --minimum-annotated-samples 60